# Image Classification System Using Convolutional Neural Networks
**Dataset:** CIFAR-10

This notebook builds a simple Convolutional Neural Network (CNN) that classifies 32x32 color images into 10 categories using the CIFAR-10 dataset (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck). The workflow follows the standard supervised deep learning pipeline: load data -> preprocess -> split -> build model -> train -> evaluate -> visualize results.

## 1. Import Libraries

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# Class names in the order CIFAR-10 uses internally
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

## 2. Load CIFAR-10 Dataset
Keras ships CIFAR-10 as two tuples: a training set and a test set.

In [3]:
(x_train_full, y_train_full), (x_test, y_test) = cifar10.load_data()

   573440/170498071 ━━━━━━━━━━━━━━━━━━━━ 1:33:03 33us/step

KeyboardInterrupt: 

## 3. Explore the Dataset

In [ ]:
print("Training images shape:", x_train_full.shape)
print("Training labels shape:", y_train_full.shape)
print("Test images shape:", x_test.shape)
print("Test labels shape:", y_test.shape)
print("Pixel value range:", x_train_full.min(), "to", x_train_full.max())

## 4. Display a Few Sample Images

In [ ]:
plt.figure(figsize=(8, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train_full[i])
    plt.title(CLASS_NAMES[y_train_full[i][0]])
    plt.axis('off')
plt.suptitle("Sample CIFAR-10 Images")
plt.tight_layout()
plt.show()

## 5. Check Dataset Shapes
Each image is 32x32 pixels with 3 color channels (RGB).

In [ ]:
print("Single image shape:", x_train_full[0].shape)
print("Number of classes:", len(CLASS_NAMES))

## 6. Preprocess the Images
Pixel values range from 0-255. Dividing by 255.0 scales them to 0-1, which helps the network train faster and more stably.

In [ ]:
x_train_full = x_train_full.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

## 7. Split Training Data into Training and Validation Sets
We hold out 20% of the training images to monitor performance on data the model does not directly learn from.

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.2, random_state=42
)

print("Training set:", x_train.shape)
print("Validation set:", x_val.shape)

## 8. Prepare/Encode Labels
Labels come in as integers 0-9. For softmax + categorical crossentropy we one-hot encode them into 10-length vectors.

In [ ]:
y_train_cat = to_categorical(y_train, num_classes=10)
y_val_cat = to_categorical(y_val, num_classes=10)
y_test_cat = to_categorical(y_test, num_classes=10)

## 9. Build a CNN Model

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation='relu', padding='same'),

    Flatten(),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

## 10. Print Model Summary

In [ ]:
model.summary()

## 11. Explain the Architecture

```
Input (32,32,3) -> Conv2D(32) -> MaxPool -> Conv2D(64) -> MaxPool
-> Conv2D(64) -> Flatten -> Dense(64) -> Dense(10, softmax)
```

Each Conv2D layer extracts visual features (edges, textures, shapes). Pooling shrinks the spatial size to reduce computation and overfitting. Flatten converts the 3D feature maps into a 1D vector so Dense layers can use them. The final Dense(10) layer outputs a probability for each of the 10 classes.

## 12. Compile the Model

In [ ]:
model.compile(
    optimizer=Adam(),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## 13. Train the Model

In [ ]:
EPOCHS = 15
BATCH_SIZE = 64

history = model.fit(
    x_train, y_train_cat,
    validation_data=(x_val, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

## 14. Plot Training and Validation Accuracy

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

## 15. Plot Training and Validation Loss

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.tight_layout()
plt.show()

## 16. Evaluate on the Test Set

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test_cat)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

## 17. Generate Predictions

In [ ]:
predictions = model.predict(x_test)
predicted_labels = np.argmax(predictions, axis=1)
true_labels = y_test.flatten()

## 18. Display a Few Test Images with Predicted and Actual Labels

In [ ]:
plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[i])
    color = 'green' if predicted_labels[i] == true_labels[i] else 'red'
    plt.title(f"P: {CLASS_NAMES[predicted_labels[i]]}\nA: {CLASS_NAMES[true_labels[i]]}",
              color=color, fontsize=8)
    plt.axis('off')
plt.suptitle("Predictions vs Actual Labels")
plt.tight_layout()
plt.show()

## 19. Create a Simple Confusion Matrix

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)
plt.figure(figsize=(8, 6))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix')
plt.colorbar()
plt.xticks(range(10), CLASS_NAMES, rotation=90)
plt.yticks(range(10), CLASS_NAMES)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()
print("Confusion Matrix:\n", cm)

## 20. Conclusion

In [ ]:
print("--- Conclusion ---")
print(f"The CNN achieved a test accuracy of {test_accuracy:.2%} on CIFAR-10.")
print("The model correctly learns basic visual patterns such as shapes,")
print("colors, and textures, though some visually similar classes")
print("(e.g., cat vs dog, automobile vs truck) are more often confused,")
print("as shown in the confusion matrix above.")